# mixup reproduction: pilot first

This notebook follows the submitted proposal. It does not contain experimental results.
Run it on a Colab GPU runtime, from VS Code connected to Colab (used for the 2026-10-28
pilot) or in the Colab browser interface. If the Drive cell fails from VS Code, use the browser.
Start with **Runtime > Change runtime type > GPU**. The pilot uses only development data.
Keep the full experiment and final-test switches off until you understand their consequences.

Put `mixup-colab-project.zip` in the top level of your Google Drive (`MyDrive`) so every new
session can reuse it. If it is not there, the next cell asks you to upload it instead.
It contains the project code, config, source attribution and tests. It contains no data,
model checkpoints, or personal PDFs.


In [18]:
#Environment check  

from pathlib import Path
import os, zipfile, subprocess, sys
import torch, torchvision, numpy
print("Python:", sys.version)
print("PyTorch:", torch.__version__, "torchvision:", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Select a GPU runtime before continuing."


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.11.0+cu130 torchvision: 0.26.0+cu130
GPU: Tesla T4


In [19]:
#Load project from Google Drive

from google.colab import drive
drive.mount("/content/drive")
ZIP = Path("/content/drive/MyDrive/mixup-colab-project.zip")
if not ZIP.exists():
    from google.colab import files
    uploaded = files.upload()
    assert "mixup-colab-project.zip" in uploaded, "Upload the project ZIP supplied with this notebook."
    ZIP = Path("mixup-colab-project.zip").resolve()
project = Path("/content/mixup-project")
project.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP) as archive:
    for item in archive.infolist():
        destination = (project / item.filename).resolve()
        assert destination.is_relative_to(project.resolve()), "Unsafe archive path"
    archive.extractall(project)
os.chdir(project)
print("Project:", project, "from", ZIP)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project: /content/mixup-project from /content/drive/MyDrive/mixup-colab-project.zip


## Save checkpoints outside the temporary runtime

The dataset lives in `/content` and can be downloaded again after a disconnect.
Runs and logs live on Drive. Use a new experiment directory for a revised protocol.
Do not run two copies of this notebook against the same output directory simultaneously.
Each checkpoint is about 135 MB. It is replaced every 5 epochs (`--save-every`) and at the
final epoch; a disconnect reruns the unsaved epochs identically. After the pilot, check that
Drive's trash is not filling up with replaced checkpoints (it counts toward your quota).


In [20]:
#Set up paths for running the project and testing it. 

RUN_ROOT = Path("/content/drive/MyDrive/mixup-reproduction-v1")
DATA = Path("/content/cifar10")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
def run(*args):
    # Stream output so test results and epoch lines appear in Colab while they run.
    with subprocess.Popen([sys.executable, "-u", *map(str, args)], stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True) as process:
        for line in process.stdout:
            print(line, end="")
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, args)
run("-m", "unittest", "discover", "-s", "tests", "-v")

test_epoch_boundaries_and_tie_rule (test_protocol.ProtocolTests.test_epoch_boundaries_and_tie_rule) ... ok
test_evaluation_gate_and_validation_selected_weights (test_protocol.ProtocolTests.test_evaluation_gate_and_validation_selected_weights) ... erm seed=1 epoch=1/1 val_error=37.50% lr=0.1 compute=0.0s [checkpoint saved]
mixup seed=1 epoch=1/1 val_error=37.50% lr=0.1 compute=0.0s [checkpoint saved]
Frozen all runs. Evaluation may now load the test dataset.
Saved final test metrics. All values are from selected validation checkpoints.
ok
test_freeze_rejects_incomplete_runs_and_blocks_later_training (test_protocol.ProtocolTests.test_freeze_rejects_incomplete_runs_and_blocks_later_training) ... erm seed=1 epoch=1/2 val_error=37.50% lr=0.1 compute=0.0s [checkpoint saved]
mixup seed=1 epoch=1/2 val_error=37.50% lr=0.1 compute=0.0s [checkpoint saved]
erm seed=1 epoch=2/2 val_error=37.50% lr=0.01 compute=0.0s [checkpoint saved]
mixup seed=1 epoch=2/2 val_error=37.50% lr=0.01 compute=0.0s [ch

## Read the protocol before running the pilot

The default is 50 epochs, batch 128, alpha 1, paired seeds 1/2/3, and a 45,000/5,000
stratified train/validation split. At this stage we only test feasibility. No test metrics are used.


In [4]:
import json
config = json.loads(Path("configs/proposal.json").read_text())
print(json.dumps(config, indent=2))
print(Path("docs/PROTOCOL.md").read_text())


{
  "experiment": "mixup-cifar10-proposal-v1",
  "epochs": 50,
  "batch_size": 128,
  "lr": 0.1,
  "momentum": 0.9,
  "weight_decay": 0.0001,
  "milestones": [
    25,
    38
  ],
  "alpha": 1.0,
  "seeds": [
    1,
    2,
    3
  ],
  "split_seed": 2026,
  "validation_per_class": 500,
  "num_workers": 2,
  "mean": [
    0.4914,
    0.4822,
    0.4465
  ],
  "std": [
    0.2023,
    0.1994,
    0.201
  ]
}
# Experiment protocol

Target: Section 3.2, Figure 3(a), PreAct ResNet-18 on CIFAR-10 in Zhang et al. (ICLR 2018).
Published errors 5.6% ERM / 4.2% mixup are reference values, never measured outputs here.

The submitted proposal is the authority for this implementation. The defaults are:

| Item | Decision |
|---|---|
| Dataset | CIFAR-10, 45,000 train / 5,000 validation / 10,000 test |
| Split | Stratified, 500 validation images per class, split seed 2026 |
| Model | Official repository ResNet18 constructor (PreActBlock), unchanged |
| Augmentation | Random crop 32 with padding 4, h

In [5]:
RUN_PILOT = True
if RUN_PILOT:
    run("-m", "mixup_repro.experiment", "pilot", "--download", "--resume",
        "--data", DATA, "--output", RUN_ROOT)


erm seed=1 epoch=1/2 val_error=87.50% lr=0.1 compute=2.1s
erm seed=1 epoch=2/2 val_error=82.42% lr=0.1 compute=1.4s [checkpoint saved]
mixup seed=1 epoch=1/2 val_error=88.67% lr=0.1 compute=1.5s
mixup seed=1 epoch=2/2 val_error=89.45% lr=0.1 compute=1.5s [checkpoint saved]
Pilot only: 1,024 train / 256 validation images. These are NOT reproduction results.


In [6]:
import csv
for method in ("erm", "mixup"):
    history = RUN_ROOT / "pilot" / f"{method}_seed1" / f"{method}_seed1_history.csv"
    if history.exists():
        print(method, list(csv.DictReader(history.open())))
print("Pilot timings exclude checkpoint writes. The full dataset is much larger.")
print("Bring these timing and GPU-memory outputs back before committing to the full budget.")


erm [{'epoch': '1', 'lr': '0.1', 'train_objective': '2.4833960831165314', 'validation_loss': '2.2909175157546997', 'validation_error_pct': '87.5', 'compute_seconds': '2.0940094890011096', 'peak_gpu_mb': '850.2685546875'}, {'epoch': '2', 'lr': '0.1', 'train_objective': '2.25424724817276', 'validation_loss': '3.552505612373352', 'validation_error_pct': '82.421875', 'compute_seconds': '1.4231928379995225', 'peak_gpu_mb': '850.2685546875'}]
mixup [{'epoch': '1', 'lr': '0.1', 'train_objective': '2.7192161083221436', 'validation_loss': '2.3524011373519897', 'validation_error_pct': '88.671875', 'compute_seconds': '1.5389657610012364', 'peak_gpu_mb': '853.01953125'}, {'epoch': '2', 'lr': '0.1', 'train_objective': '9.338361859321594', 'validation_loss': '9165937152.0', 'validation_error_pct': '89.453125', 'compute_seconds': '1.4528490100001363', 'peak_gpu_mb': '852.76953125'}]
Pilot timings exclude checkpoint writes. The full dataset is much larger.
Bring these timing and GPU-memory outputs bac

## Full reproduction (off by default)

Proceed after reviewing pilot feasibility. The full experiment needs six runs across one or more Colab sessions.
Set one METHOD/SEED pair at a time for easier supervision. Resume does not reset completed epochs.
If code/config changes, use a newly documented protocol and output directory.


In [21]:
#Training 
RUN_TRAINING = True
METHOD = "mixup"  # start with erm then "mixup"; repeat both for seeds 2 and 3
SEED = 1
MAIN = RUN_ROOT / "main"
if RUN_TRAINING:
    run("-m", "mixup_repro.experiment", "train", "--method", METHOD, "--seed", SEED,
        "--download", "--resume", "--save-every", 5, "--data", DATA, "--output", MAIN)


mixup seed=1 epoch=6/50 val_error=61.54% lr=0.1 compute=57.9s
mixup seed=1 epoch=7/50 val_error=59.38% lr=0.1 compute=58.8s
mixup seed=1 epoch=8/50 val_error=58.44% lr=0.1 compute=59.0s
mixup seed=1 epoch=9/50 val_error=58.86% lr=0.1 compute=58.6s
mixup seed=1 epoch=10/50 val_error=52.66% lr=0.1 compute=58.9s [checkpoint saved]
mixup seed=1 epoch=11/50 val_error=51.12% lr=0.1 compute=59.1s
mixup seed=1 epoch=12/50 val_error=47.14% lr=0.1 compute=59.0s
mixup seed=1 epoch=13/50 val_error=45.82% lr=0.1 compute=58.9s
mixup seed=1 epoch=14/50 val_error=44.40% lr=0.1 compute=58.8s
mixup seed=1 epoch=15/50 val_error=45.70% lr=0.1 compute=59.0s [checkpoint saved]
mixup seed=1 epoch=16/50 val_error=39.44% lr=0.1 compute=59.2s
mixup seed=1 epoch=17/50 val_error=36.88% lr=0.1 compute=59.2s
mixup seed=1 epoch=18/50 val_error=38.88% lr=0.1 compute=59.0s
mixup seed=1 epoch=19/50 val_error=40.84% lr=0.1 compute=59.1s
mixup seed=1 epoch=20/50 val_error=35.36% lr=0.1 compute=59.0s [checkpoint saved]
mi

## Freeze and final evaluation (off by default)

Only proceed after all six runs finish and every decision is fixed. Freeze verifies matching
configs, splits, source hashes, completion and paired initialization. It blocks further training.
The evaluator loads only validation-selected model weights and evaluates the untouched test split.


In [8]:
FREEZE_PROTOCOL = False
EVALUATE_TEST = False
if FREEZE_PROTOCOL:
    run("-m", "mixup_repro.experiment", "freeze", "--output", MAIN)
if EVALUATE_TEST:
    run("-m", "mixup_repro.experiment", "evaluate", "--download", "--data", DATA, "--output", MAIN)


In [9]:
if (MAIN / "test_results.json").exists():
    run("-m", "mixup_repro.summarize", "--output", MAIN)
    from IPython.display import Image, display, Markdown
    display(Markdown((MAIN / "summary.md").read_text()))
    display(Image(filename=str(MAIN / "learning_curves.png")))
else:
    print("No final test results yet. No scientific result is inferred from the pilot.")


No final test results yet. No scientific result is inferred from the pilot.
